## Importación de librerías necesarias para el procesamiento de datos

In [ ]:
# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import os
import re
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    LSTM,
    Dense,
    Dropout
)



import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)
# Importar librerías para visualización de datos y métricas de evaluación
import seaborn as sns                    # Visualizaciones estadísticas elegantes
from time import time

In [ ]:
# ============================================================
# REPRODUCIBILIDAD DEL EXPERIMENTO
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print("✅ Semilla fijada:", SEED)

✅ Semilla fijada: 42


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# CARGAR DEVELOPMENT DATASET V2
# PARA DESARROLLO CNN-LSTM V3
# ============================================================

import os
import numpy as np

DATA_PATH = (
    "/content/drive/MyDrive/"
    "CAN_IDS_V3/"
    "development_v2_train_validation.npz"
)

if not os.path.isfile(DATA_PATH):

    raise FileNotFoundError(
        f"No se encontró:\n{DATA_PATH}"
    )


data_v3 = np.load(
    DATA_PATH
)

X_train = data_v3["X_train"]
y_train = data_v3["y_train"]

X_val = data_v3["X_val"]
y_val = data_v3["y_val"]


print(
    "=============================================="
)

print(
    "V3 DEVELOPMENT DATA LOADED"
)

print(
    "=============================================="
)

print(
    "X_train:",
    X_train.shape
)

print(
    "y_train:",
    y_train.shape
)

print(
    "X_val:",
    X_val.shape
)

print(
    "y_val:",
    y_val.shape
)


# ------------------------------------------------------------
# Assertions de seguridad
# ------------------------------------------------------------

assert X_train.shape == (
    147628,
    20,
    10
)

assert y_train.shape == (
    147628,
)

assert X_val.shape == (
    36907,
    20,
    10
)

assert y_val.shape == (
    36907,
)


print(
    "\n✅ Exact V2 Training/Validation "
    "restored for V3 development."
)

V3 DEVELOPMENT DATA LOADED
X_train: (147628, 20, 10)
y_train: (147628,)
X_val: (36907, 20, 10)
y_val: (36907,)

✅ Exact V2 Training/Validation restored for V3 development.


In [ ]:
# ============================================================
# CARGAR TRAINING + VALIDATION V2 PARA DESARROLLO V3
# ============================================================

from google.colab import drive
import os
import numpy as np

# Montar Drive
drive.mount(
    "/content/drive",
    force_remount=False
)

# Archivo creado previamente desde modelo_2026V2.ipynb
DATA_PATH = (
    "/content/drive/MyDrive/"
    "CAN_IDS_V3/"
    "development_v2_train_validation.npz"
)

print("Ruta:")
print(DATA_PATH)

print(
    "\n¿Existe el archivo?:",
    os.path.isfile(DATA_PATH)
)

# ------------------------------------------------------------
# Si no existe, detener aquí
# ------------------------------------------------------------

if not os.path.isfile(DATA_PATH):

    raise FileNotFoundError(
        "\nNo existe development_v2_train_validation.npz.\n"
        "Debes crearlo primero desde modelo_2026V2.ipynb."
    )


# ------------------------------------------------------------
# Cargar datos
# ------------------------------------------------------------

data_v3 = np.load(
    DATA_PATH
)

X_train = data_v3["X_train"]
y_train = data_v3["y_train"]

X_val = data_v3["X_val"]
y_val = data_v3["y_val"]


# ------------------------------------------------------------
# Verificación
# ------------------------------------------------------------

print("\n==============================================")
print("V3 DEVELOPMENT DATA LOADED")
print("==============================================")

print(
    "X_train:",
    X_train.shape
)

print(
    "y_train:",
    y_train.shape
)

print(
    "X_val:",
    X_val.shape
)

print(
    "y_val:",
    y_val.shape
)


# ------------------------------------------------------------
# Assertions del dataset V2 definitivo
# ------------------------------------------------------------

assert X_train.shape == (
    147628,
    20,
    10
)

assert y_train.shape == (
    147628,
)

assert X_val.shape == (
    36907,
    20,
    10
)

assert y_val.shape == (
    36907,
)


print(
    "\n✅ Training y Validation V2 "
    "cargados correctamente para V3."
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ruta:
/content/drive/MyDrive/CAN_IDS_V3/development_v2_train_validation.npz

¿Existe el archivo?: True

V3 DEVELOPMENT DATA LOADED
X_train: (147628, 20, 10)
y_train: (147628,)
X_val: (36907, 20, 10)
y_val: (36907,)

✅ Training y Validation V2 cargados correctamente para V3.


## CONSTRUCCIÓN DEL MODELO HÍBRIDO CNN-LSTM

In [ ]:
# ============================================================
# CNN-LSTM V3-A
# DEVELOPMENT: TRAINING + VALIDATION ONLY
# TEST SET NOT USED
# ============================================================

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    Activation,
    MaxPooling1D,
    SpatialDropout1D,
    LSTM,
    Dropout,
    Dense
)

# ============================================================
# REPRODUCIBILIDAD
# ============================================================

SEED = 42

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


# ============================================================
# DIMENSIONES
# ============================================================

WINDOW_SIZE = 20
N_FEATURES = 10
NUM_CLASSES = 4


# ============================================================
# ARQUITECTURA CNN-LSTM V3-A
# ============================================================

model_v3a = Sequential([

    Input(
        shape=(WINDOW_SIZE, N_FEATURES)
    ),

    # --------------------------------------------------------
    # CNN Block 1
    # --------------------------------------------------------

    Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        use_bias=False
    ),

    BatchNormalization(),

    Activation(
        "relu"
    ),


    # --------------------------------------------------------
    # CNN Block 2
    # --------------------------------------------------------

    Conv1D(
        filters=128,
        kernel_size=3,
        padding="same",
        use_bias=False
    ),

    BatchNormalization(),

    Activation(
        "relu"
    ),


    # --------------------------------------------------------
    # Temporal dimensionality reduction
    # --------------------------------------------------------

    MaxPooling1D(
        pool_size=2
    ),

    SpatialDropout1D(
        rate=0.10
    ),


    # --------------------------------------------------------
    # Sequential modeling
    # --------------------------------------------------------

    LSTM(
        units=64,
        return_sequences=False
    ),

    Dropout(
        rate=0.20
    ),


    # --------------------------------------------------------
    # Classification head
    # --------------------------------------------------------

    Dense(
        units=32,
        activation="relu"
    ),

    Dropout(
        rate=0.10
    ),

    Dense(
        units=NUM_CLASSES,
        activation="softmax"
    )
])


# ============================================================
# SUMMARY
# ============================================================

model_v3a.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 20, 64)         │         1,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 20, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 20, 128)        │        24,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 20, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 20, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 10, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 10, 128)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 78,884 (308.14 KB)

 Trainable params: 78,500 (306.64 KB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
# ============================================================
# COMPILACIÓN + CALLBACKS CNN-LSTM V3-A
# ============================================================

import tensorflow as tf

optimizer_v3a = tf.keras.optimizers.AdamW(
    learning_rate=3e-4,
    weight_decay=1e-4
)

model_v3a.compile(
    optimizer=optimizer_v3a,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stopping_v3a = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_v3a = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

print("✅ CNN-LSTM V3-A compilado.")
print("✅ EarlyStopping creado.")
print("✅ ReduceLROnPlateau creado.")

✅ CNN-LSTM V3-A compilado.
✅ EarlyStopping creado.
✅ ReduceLROnPlateau creado.


In [ ]:
print("model_v3a existe:", "model_v3a" in globals())
print("early_stopping_v3a existe:", "early_stopping_v3a" in globals())
print("reduce_lr_v3a existe:", "reduce_lr_v3a" in globals())
print("X_train existe:", "X_train" in globals())
print("X_val existe:", "X_val" in globals())

model_v3a existe: True
early_stopping_v3a existe: True
reduce_lr_v3a existe: True
X_train existe: True
X_val existe: True


**entrenamiento**

In [ ]:
from time import perf_counter

EPOCHS_MAX = 30
BATCH_SIZE = 64

start_v3a = perf_counter()

history_v3a = model_v3a.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS_MAX,
    batch_size=BATCH_SIZE,
    callbacks=[
        early_stopping_v3a,
        reduce_lr_v3a
    ],
    verbose=1
)

v3a_training_time = perf_counter() - start_v3a

print("\n==============================================")
print("CNN-LSTM V3-A TRAINING")
print("==============================================")

print(f"Training Time: {v3a_training_time:.4f} s")
print(
    "Epochs executed:",
    len(history_v3a.history["loss"])
)

Epoch 1/30
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 76s 31ms/step - accuracy: 0.9582 - loss: 0.1101 - val_accuracy: 0.9976 - val_loss: 0.0080 - learning_rate: 3.0000e-04
Epoch 2/30
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 68s 29ms/step - accuracy: 0.9976 - loss: 0.0082 - val_accuracy: 0.9968 - val_loss: 0.0100 - learning_rate: 3.0000e-04
Epoch 3/30
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 69s 30ms/step - accuracy: 0.9987 - loss: 0.0041 - val_accuracy: 0.9993 - val_loss: 0.0021 - learning_rate: 3.0000e-04
Epoch 4/30
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 69s 30ms/step - accuracy: 0.9992 - loss: 0.0026 - val_accuracy: 0.9992 - val_loss: 0.0025 - learning_rate: 3.0000e-04
Epoch 5/30
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 70s 30ms/step - accuracy: 0.9994 - loss: 0.0019 - val_accuracy: 0.9997 - val_loss: 0.0011 - learning_rate: 3.0000e-04
Epoch 6/30
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 81s 30ms/step - accuracy: 0.9995 - loss: 0.0014 - val_accuracy: 0.9998 - val_loss: 0.0011 - learning_rate: 3.0000e-04
Epoch 7/30
2305/2307 ━━━━━━━━━━━━━━━━━━━

In [ ]:
# ============================================================
# CNN-LSTM V3-A — VALIDATION RESULTS
# NO USA TEST
# ============================================================

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# Predicción sobre Validation
# ------------------------------------------------------------

val_prob_v3a = model_v3a.predict(
    X_val,
    batch_size=64,
    verbose=0
)

val_pred_v3a = np.argmax(
    val_prob_v3a,
    axis=1
)

# ------------------------------------------------------------
# Métricas
# ------------------------------------------------------------

val_accuracy_v3a = accuracy_score(
    y_val,
    val_pred_v3a
)

val_macro_precision_v3a = precision_score(
    y_val,
    val_pred_v3a,
    average="macro",
    zero_division=0
)

val_macro_recall_v3a = recall_score(
    y_val,
    val_pred_v3a,
    average="macro",
    zero_division=0
)

val_macro_f1_v3a = f1_score(
    y_val,
    val_pred_v3a,
    average="macro",
    zero_division=0
)

val_weighted_f1_v3a = f1_score(
    y_val,
    val_pred_v3a,
    average="weighted",
    zero_division=0
)

val_errors_v3a = int(
    np.sum(
        y_val != val_pred_v3a
    )
)

val_correct_v3a = (
    len(y_val)
    - val_errors_v3a
)

# ------------------------------------------------------------
# Resultados
# ------------------------------------------------------------

print("\n==============================================")
print("CNN-LSTM V3-A — VALIDATION RESULTS")
print("==============================================")

print(
    f"Validation sequences: {len(y_val):,}"
)

print(
    f"Correct predictions:  {val_correct_v3a:,}"
)

print(
    f"Misclassifications:   {val_errors_v3a:,}"
)

print("\n----------------------------------------------")

print(
    f"Accuracy:        {val_accuracy_v3a*100:.6f}%"
)

print(
    f"Macro Precision: {val_macro_precision_v3a*100:.6f}%"
)

print(
    f"Macro Recall:    {val_macro_recall_v3a*100:.6f}%"
)

print(
    f"Macro F1:        {val_macro_f1_v3a*100:.6f}%"
)

print(
    f"Weighted F1:     {val_weighted_f1_v3a*100:.6f}%"
)

# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print("\n==============================================")
print("CLASSIFICATION REPORT — VALIDATION")
print("==============================================")

print(
    classification_report(
        y_val,
        val_pred_v3a,
        digits=6,
        zero_division=0
    )
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

print("\n==============================================")
print("CONFUSION MATRIX — VALIDATION")
print("==============================================")

cm_v3a_val = confusion_matrix(
    y_val,
    val_pred_v3a
)

print(cm_v3a_val)


CNN-LSTM V3-A — VALIDATION RESULTS
Validation sequences: 36,907
Correct predictions:  36,900
Misclassifications:   7

----------------------------------------------
Accuracy:        99.981033%
Macro Precision: 99.984988%
Macro Recall:    99.971025%
Macro F1:        99.978004%
Weighted F1:     99.981032%

CLASSIFICATION REPORT — VALIDATION
              precision    recall  f1-score   support

           0   0.999736  1.000000  0.999868     18955
           1   1.000000  0.999429  0.999714      5253
           2   0.999789  0.999789  0.999789      4736
           3   0.999874  0.999623  0.999749      7963

    accuracy                       0.999810     36907
   macro avg   0.999850  0.999710  0.999780     36907
weighted avg   0.999810  0.999810  0.999810     36907


CONFUSION MATRIX — VALIDATION
[[18955     0     0     0]
 [    1  5250     1     1]
 [    1     0  4735     0]
 [    3     0     0  7960]]


**Primero: guarda V3-A inmediatamente**

In [ ]:
# ============================================================
# GUARDAR CANDIDATA CNN-LSTM V3-A
# ============================================================

import os
import json
import numpy as np

SAVE_DIR = "/content/drive/MyDrive/CAN_IDS_V3/V3A"
os.makedirs(SAVE_DIR, exist_ok=True)

# Modelo completo
model_v3a.save(
    os.path.join(
        SAVE_DIR,
        "cnn_lstm_v3a.keras"
    )
)

# Predicciones de VALIDATION
np.savez_compressed(
    os.path.join(
        SAVE_DIR,
        "v3a_validation_results.npz"
    ),
    y_val=y_val,
    val_pred=val_pred_v3a,
    val_prob=val_prob_v3a
)

# Métricas
metrics_v3a = {
    "validation_sequences": int(len(y_val)),
    "correct_predictions": int(len(y_val) - val_errors_v3a),
    "misclassifications": int(val_errors_v3a),
    "accuracy": float(val_accuracy_v3a),
    "macro_precision": float(val_macro_precision_v3a),
    "macro_recall": float(val_macro_recall_v3a),
    "macro_f1": float(val_macro_f1_v3a),
    "weighted_f1": float(val_weighted_f1_v3a),
    "training_time_seconds": float(v3a_training_time),
    "epochs_executed": int(len(history_v3a.history["loss"])),
    "selected_by": "validation loss with EarlyStopping",
    "test_used": False
}

with open(
    os.path.join(
        SAVE_DIR,
        "v3a_metrics.json"
    ),
    "w"
) as f:
    json.dump(
        metrics_v3a,
        f,
        indent=4
    )

print("✅ CNN-LSTM V3-A guardada.")

✅ CNN-LSTM V3-A guardada.


**Construir CNN–LSTM V3-B**

In [ ]:
# ============================================================
# CNN-LSTM V3-B
# Conv1 = 64
# Conv2 = 64
# LSTM  = 64
# Dense = 32
#
# DEVELOPMENT: TRAINING + VALIDATION ONLY
# TEST SET NOT USED
# ============================================================

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    Activation,
    MaxPooling1D,
    SpatialDropout1D,
    LSTM,
    Dropout,
    Dense
)

# ============================================================
# REPRODUCIBILIDAD
# ============================================================

SEED = 42

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


# ============================================================
# DIMENSIONES
# ============================================================

WINDOW_SIZE = 20
N_FEATURES = 10
NUM_CLASSES = 4


# ============================================================
# ARQUITECTURA CNN-LSTM V3-B
# ============================================================

model_v3b = Sequential([

    Input(
        shape=(WINDOW_SIZE, N_FEATURES)
    ),

    # --------------------------------------------------------
    # CNN Block 1
    # --------------------------------------------------------

    Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        use_bias=False
    ),

    BatchNormalization(),

    Activation("relu"),


    # --------------------------------------------------------
    # CNN Block 2
    # DIFERENCIA PRINCIPAL RESPECTO V3-A:
    # 64 filtros en lugar de 128
    # --------------------------------------------------------

    Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        use_bias=False
    ),

    BatchNormalization(),

    Activation("relu"),


    # --------------------------------------------------------
    # Temporal reduction
    # --------------------------------------------------------

    MaxPooling1D(
        pool_size=2
    ),

    SpatialDropout1D(
        rate=0.10
    ),


    # --------------------------------------------------------
    # Sequential modeling
    # --------------------------------------------------------

    LSTM(
        units=64,
        return_sequences=False
    ),

    Dropout(
        rate=0.20
    ),


    # --------------------------------------------------------
    # Classification head
    # --------------------------------------------------------

    Dense(
        units=32,
        activation="relu"
    ),

    Dropout(
        rate=0.10
    ),

    Dense(
        units=NUM_CLASSES,
        activation="softmax"
    )
])


# ============================================================
# SUMMARY
# ============================================================

model_v3b.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 20, 64)         │         1,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 20, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 20, 64)         │        12,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 20, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 10, 64)         │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49,956 (195.14 KB)

 Trainable params: 49,700 (194.14 KB)

 Non-trainable params: 256 (1.00 KB)

In [ ]:
# ============================================================
# COMPILACIÓN + CALLBACKS CNN-LSTM V3-B
# ============================================================

optimizer_v3b = tf.keras.optimizers.AdamW(
    learning_rate=3e-4,
    weight_decay=1e-4
)

model_v3b.compile(
    optimizer=optimizer_v3b,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


early_stopping_v3b = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
    verbose=1
)


reduce_lr_v3b = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)


print("✅ CNN-LSTM V3-B compilado.")
print("✅ EarlyStopping V3-B creado.")
print("✅ ReduceLROnPlateau V3-B creado.")

✅ CNN-LSTM V3-B compilado.
✅ EarlyStopping V3-B creado.
✅ ReduceLROnPlateau V3-B creado.


In [ ]:
# ============================================================
# TRAIN CNN-LSTM V3-B
# TRAINING + VALIDATION ONLY
# ============================================================

from time import perf_counter

EPOCHS_MAX = 30
BATCH_SIZE = 64


start_v3b = perf_counter()


history_v3b = model_v3b.fit(

    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=EPOCHS_MAX,

    batch_size=BATCH_SIZE,

    callbacks=[
        early_stopping_v3b,
        reduce_lr_v3b
    ],

    verbose=1
)


v3b_training_time = (
    perf_counter()
    - start_v3b
)


print("\n==============================================")
print("CNN-LSTM V3-B TRAINING")
print("==============================================")

print(
    f"Training Time: "
    f"{v3b_training_time:.4f} s"
)

print(
    f"Epochs executed: "
    f"{len(history_v3b.history['loss'])}"
)

Epoch 1/30
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 52s 21ms/step - accuracy: 0.9478 - loss: 0.1363 - val_accuracy: 0.9974 - val_loss: 0.0093 - learning_rate: 3.0000e-04
Epoch 2/30
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 48s 21ms/step - accuracy: 0.9951 - loss: 0.0149 - val_accuracy: 0.9950 - val_loss: 0.0147 - learning_rate: 3.0000e-04
Epoch 3/30
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 47s 20ms/step - accuracy: 0.9983 - loss: 0.0056 - val_accuracy: 0.9995 - val_loss: 0.0017 - learning_rate: 3.0000e-04
Epoch 4/30
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 47s 20ms/step - accuracy: 0.9989 - loss: 0.0033 - val_accuracy: 0.9995 - val_loss: 0.0017 - learning_rate: 3.0000e-04
Epoch 5/30
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 49s 21ms/step - accuracy: 0.9993 - loss: 0.0022 - val_accuracy: 0.9996 - val_loss: 0.0014 - learning_rate: 3.0000e-04
Epoch 6/30
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 82s 21ms/step - accuracy: 0.9995 - loss: 0.0017 - val_accuracy: 0.9993 - val_loss: 0.0024 - learning_rate: 3.0000e-04
Epoch 7/30
2306/2307 ━━━━━━━━━━━━━━━━━━━

In [ ]:
# ============================================================
# CNN-LSTM V3-B — VALIDATION RESULTS
# TEST SET IS NOT USED
# ============================================================

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# PREDICCIONES
# ============================================================

val_prob_v3b = model_v3b.predict(
    X_val,
    batch_size=64,
    verbose=0
)

val_pred_v3b = np.argmax(
    val_prob_v3b,
    axis=1
)


# ============================================================
# MÉTRICAS
# ============================================================

val_accuracy_v3b = accuracy_score(
    y_val,
    val_pred_v3b
)

val_macro_precision_v3b = precision_score(
    y_val,
    val_pred_v3b,
    average="macro",
    zero_division=0
)

val_macro_recall_v3b = recall_score(
    y_val,
    val_pred_v3b,
    average="macro",
    zero_division=0
)

val_macro_f1_v3b = f1_score(
    y_val,
    val_pred_v3b,
    average="macro",
    zero_division=0
)

val_weighted_f1_v3b = f1_score(
    y_val,
    val_pred_v3b,
    average="weighted",
    zero_division=0
)


val_errors_v3b = int(
    np.sum(
        val_pred_v3b != y_val
    )
)

val_correct_v3b = (
    len(y_val)
    - val_errors_v3b
)


# ============================================================
# RESULTADOS
# ============================================================

print("\n==============================================")
print("CNN-LSTM V3-B — VALIDATION RESULTS")
print("==============================================")

print(
    f"Validation sequences: {len(y_val):,}"
)

print(
    f"Correct predictions:  {val_correct_v3b:,}"
)

print(
    f"Misclassifications:   {val_errors_v3b:,}"
)

print("\n----------------------------------------------")

print(
    f"Accuracy:        "
    f"{val_accuracy_v3b*100:.6f}%"
)

print(
    f"Macro Precision: "
    f"{val_macro_precision_v3b*100:.6f}%"
)

print(
    f"Macro Recall:    "
    f"{val_macro_recall_v3b*100:.6f}%"
)

print(
    f"Macro F1:        "
    f"{val_macro_f1_v3b*100:.6f}%"
)

print(
    f"Weighted F1:     "
    f"{val_weighted_f1_v3b*100:.6f}%"
)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n==============================================")
print("CLASSIFICATION REPORT — V3-B VALIDATION")
print("==============================================")

print(
    classification_report(
        y_val,
        val_pred_v3b,
        digits=6,
        zero_division=0
    )
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm_v3b_val = confusion_matrix(
    y_val,
    val_pred_v3b
)

print("\n==============================================")
print("CONFUSION MATRIX — V3-B VALIDATION")
print("==============================================")

print(
    cm_v3b_val
)


CNN-LSTM V3-B — VALIDATION RESULTS
Validation sequences: 36,907
Correct predictions:  36,898
Misclassifications:   9

----------------------------------------------
Accuracy:        99.975614%
Macro Precision: 99.975072%
Macro Recall:    99.966248%
Macro F1:        99.970657%
Weighted F1:     99.975615%

CLASSIFICATION REPORT — V3-B VALIDATION
              precision    recall  f1-score   support

           0   0.999842  0.999894  0.999868     18955
           1   1.000000  0.999429  0.999714      5253
           2   0.999789  0.999578  0.999683      4736
           3   0.999372  0.999749  0.999561      7963

    accuracy                       0.999756     36907
   macro avg   0.999751  0.999662  0.999707     36907
weighted avg   0.999756  0.999756  0.999756     36907


CONFUSION MATRIX — V3-B VALIDATION
[[18953     0     0     2]
 [    1  5250     1     1]
 [    0     0  4734     2]
 [    2     0     0  7961]]


**Guarda V3-B inmediatamente**

In [ ]:
# ============================================================
# GUARDAR CNN-LSTM V3-B
# ============================================================

import os
import json
import numpy as np

SAVE_DIR_V3B = (
    "/content/drive/MyDrive/"
    "CAN_IDS_V3/V3B"
)

os.makedirs(
    SAVE_DIR_V3B,
    exist_ok=True
)


# Modelo
model_v3b.save(
    os.path.join(
        SAVE_DIR_V3B,
        "cnn_lstm_v3b.keras"
    )
)


# Predicciones Validation
np.savez_compressed(
    os.path.join(
        SAVE_DIR_V3B,
        "v3b_validation_results.npz"
    ),
    y_val=y_val,
    val_pred=val_pred_v3b,
    val_prob=val_prob_v3b
)


# Métricas
metrics_v3b = {

    "model": "CNN-LSTM V3-B",

    "conv1_filters": 64,
    "conv2_filters": 64,
    "lstm_units": 64,
    "dense_units": 32,

    "parameters":
        int(model_v3b.count_params()),

    "validation_sequences":
        int(len(y_val)),

    "correct_predictions":
        int(val_correct_v3b),

    "misclassifications":
        int(val_errors_v3b),

    "accuracy":
        float(val_accuracy_v3b),

    "macro_precision":
        float(val_macro_precision_v3b),

    "macro_recall":
        float(val_macro_recall_v3b),

    "macro_f1":
        float(val_macro_f1_v3b),

    "weighted_f1":
        float(val_weighted_f1_v3b),

    "training_time_seconds":
        float(v3b_training_time),

    "epochs_executed":
        int(
            len(
                history_v3b.history["loss"]
            )
        ),

    "test_used":
        False
}


with open(
    os.path.join(
        SAVE_DIR_V3B,
        "v3b_metrics.json"
    ),
    "w"
) as f:

    json.dump(
        metrics_v3b,
        f,
        indent=4
    )


print(
    "✅ CNN-LSTM V3-B guardada."
)

✅ CNN-LSTM V3-B guardada.


**Construir CNN–LSTM V3-C**

In [ ]:
# ============================================================
# CNN-LSTM V3-C
# Conv1 = 64
# Conv2 = 128
# LSTM  = 32
# Dense = 32
#
# DEVELOPMENT: TRAINING + VALIDATION ONLY
# TEST SET NOT USED
# ============================================================

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    Activation,
    MaxPooling1D,
    SpatialDropout1D,
    LSTM,
    Dropout,
    Dense
)

SEED = 42

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


WINDOW_SIZE = 20
N_FEATURES = 10
NUM_CLASSES = 4


model_v3c = Sequential([

    Input(
        shape=(WINDOW_SIZE, N_FEATURES)
    ),

    # CNN Block 1
    Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        use_bias=False
    ),

    BatchNormalization(),
    Activation("relu"),

    # CNN Block 2
    Conv1D(
        filters=128,
        kernel_size=3,
        padding="same",
        use_bias=False
    ),

    BatchNormalization(),
    Activation("relu"),

    MaxPooling1D(
        pool_size=2
    ),

    SpatialDropout1D(
        rate=0.10
    ),

    # DIFERENCIA RESPECTO V3-A:
    # 32 unidades LSTM en lugar de 64
    LSTM(
        units=32,
        return_sequences=False
    ),

    Dropout(
        rate=0.20
    ),

    Dense(
        units=32,
        activation="relu"
    ),

    Dropout(
        rate=0.10
    ),

    Dense(
        units=NUM_CLASSES,
        activation="softmax"
    )
])


model_v3c.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 20, 64)         │         1,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 20, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 20, 128)        │        24,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 20, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 20, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 10, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 10, 128)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │        20,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49,060 (191.64 KB)

 Trainable params: 48,676 (190.14 KB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
# ============================================================
# COMPILACIÓN + CALLBACKS CNN-LSTM V3-C
# ============================================================

optimizer_v3c = tf.keras.optimizers.AdamW(
    learning_rate=3e-4,
    weight_decay=1e-4
)

model_v3c.compile(
    optimizer=optimizer_v3c,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


early_stopping_v3c = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
    verbose=1
)


reduce_lr_v3c = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)


print("✅ CNN-LSTM V3-C compilado.")

✅ CNN-LSTM V3-C compilado.


In [ ]:
# ============================================================
# TRAIN CNN-LSTM V3-C
# ============================================================

from time import perf_counter

EPOCHS_MAX = 30
BATCH_SIZE = 64

start_v3c = perf_counter()

history_v3c = model_v3c.fit(

    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=EPOCHS_MAX,

    batch_size=BATCH_SIZE,

    callbacks=[
        early_stopping_v3c,
        reduce_lr_v3c
    ],

    verbose=1
)

v3c_training_time = (
    perf_counter()
    - start_v3c
)

print("\n==============================================")
print("CNN-LSTM V3-C TRAINING")
print("==============================================")

print(
    f"Training Time: {v3c_training_time:.4f} s"
)

print(
    "Epochs executed:",
    len(history_v3c.history["loss"])
)

Epoch 1/30
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 59s 24ms/step - accuracy: 0.9517 - loss: 0.1310 - val_accuracy: 0.9983 - val_loss: 0.0062 - learning_rate: 3.0000e-04
Epoch 2/30
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 56s 24ms/step - accuracy: 0.9975 - loss: 0.0083 - val_accuracy: 0.9993 - val_loss: 0.0024 - learning_rate: 3.0000e-04
Epoch 3/30
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 55s 24ms/step - accuracy: 0.9989 - loss: 0.0037 - val_accuracy: 0.9990 - val_loss: 0.0035 - learning_rate: 3.0000e-04
Epoch 4/30
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 54s 24ms/step - accuracy: 0.9992 - loss: 0.0026 - val_accuracy: 0.9996 - val_loss: 0.0013 - learning_rate: 3.0000e-04
Epoch 5/30
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 54s 23ms/step - accuracy: 0.9994 - loss: 0.0019 - val_accuracy: 0.9988 - val_loss: 0.0038 - learning_rate: 3.0000e-04
Epoch 6/30
2306/2307 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9995 - loss: 0.0016
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.
2307/2307 ━━━━━━━━━━━━━━━━━━━━ 55s 

In [ ]:
# ============================================================
# CNN-LSTM V3-C — VALIDATION RESULTS
# TEST SET IS NOT USED
# ============================================================

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# PREDICCIONES SOBRE VALIDATION
# ============================================================

val_prob_v3c = model_v3c.predict(
    X_val,
    batch_size=64,
    verbose=0
)

val_pred_v3c = np.argmax(
    val_prob_v3c,
    axis=1
)

# ============================================================
# MÉTRICAS
# ============================================================

val_accuracy_v3c = accuracy_score(
    y_val,
    val_pred_v3c
)

val_macro_precision_v3c = precision_score(
    y_val,
    val_pred_v3c,
    average="macro",
    zero_division=0
)

val_macro_recall_v3c = recall_score(
    y_val,
    val_pred_v3c,
    average="macro",
    zero_division=0
)

val_macro_f1_v3c = f1_score(
    y_val,
    val_pred_v3c,
    average="macro",
    zero_division=0
)

val_weighted_f1_v3c = f1_score(
    y_val,
    val_pred_v3c,
    average="weighted",
    zero_division=0
)

val_errors_v3c = int(
    np.sum(y_val != val_pred_v3c)
)

val_correct_v3c = (
    len(y_val) - val_errors_v3c
)

# ============================================================
# RESULTADOS
# ============================================================

print("\n==============================================")
print("CNN-LSTM V3-C — VALIDATION RESULTS")
print("==============================================")

print(f"Validation sequences: {len(y_val):,}")
print(f"Correct predictions:  {val_correct_v3c:,}")
print(f"Misclassifications:   {val_errors_v3c:,}")

print("\n----------------------------------------------")

print(f"Accuracy:        {val_accuracy_v3c*100:.6f}%")
print(f"Macro Precision: {val_macro_precision_v3c*100:.6f}%")
print(f"Macro Recall:    {val_macro_recall_v3c*100:.6f}%")
print(f"Macro F1:        {val_macro_f1_v3c*100:.6f}%")
print(f"Weighted F1:     {val_weighted_f1_v3c*100:.6f}%")

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n==============================================")
print("CLASSIFICATION REPORT — V3-C VALIDATION")
print("==============================================")

print(
    classification_report(
        y_val,
        val_pred_v3c,
        digits=6,
        zero_division=0
    )
)

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm_v3c_val = confusion_matrix(
    y_val,
    val_pred_v3c
)

print("\n==============================================")
print("CONFUSION MATRIX — V3-C VALIDATION")
print("==============================================")

print(cm_v3c_val)


CNN-LSTM V3-C — VALIDATION RESULTS
Validation sequences: 36,907
Correct predictions:  36,902
Misclassifications:   5

----------------------------------------------
Accuracy:        99.986452%
Macro Precision: 99.987945%
Macro Recall:    99.982884%
Macro F1:        99.985413%
Weighted F1:     99.986452%

CLASSIFICATION REPORT — V3-C VALIDATION
              precision    recall  f1-score   support

           0   0.999894  0.999947  0.999921     18955
           1   1.000000  0.999619  0.999810      5253
           2   1.000000  1.000000  1.000000      4736
           3   0.999623  0.999749  0.999686      7963

    accuracy                       0.999865     36907
   macro avg   0.999879  0.999829  0.999854     36907
weighted avg   0.999865  0.999865  0.999865     36907


CONFUSION MATRIX — V3-C VALIDATION
[[18954     0     0     1]
 [    0  5251     0     2]
 [    0     0  4736     0]
 [    2     0     0  7961]]


**Después guarda V3-C**

In [ ]:
# ============================================================
# GUARDAR CNN-LSTM V3-C
# ============================================================

import os
import json
import numpy as np

SAVE_DIR_V3C = (
    "/content/drive/MyDrive/"
    "CAN_IDS_V3/V3C"
)

os.makedirs(
    SAVE_DIR_V3C,
    exist_ok=True
)

# Modelo
model_v3c.save(
    os.path.join(
        SAVE_DIR_V3C,
        "cnn_lstm_v3c.keras"
    )
)

# Predicciones Validation
np.savez_compressed(
    os.path.join(
        SAVE_DIR_V3C,
        "v3c_validation_results.npz"
    ),
    y_val=y_val,
    val_pred=val_pred_v3c,
    val_prob=val_prob_v3c
)

metrics_v3c = {
    "model": "CNN-LSTM V3-C",
    "conv1_filters": 64,
    "conv2_filters": 128,
    "lstm_units": 32,
    "dense_units": 32,
    "parameters": int(model_v3c.count_params()),
    "validation_sequences": int(len(y_val)),
    "correct_predictions": int(val_correct_v3c),
    "misclassifications": int(val_errors_v3c),
    "accuracy": float(val_accuracy_v3c),
    "macro_precision": float(val_macro_precision_v3c),
    "macro_recall": float(val_macro_recall_v3c),
    "macro_f1": float(val_macro_f1_v3c),
    "weighted_f1": float(val_weighted_f1_v3c),
    "training_time_seconds": float(v3c_training_time),
    "epochs_executed": int(
        len(history_v3c.history["loss"])
    ),
    "best_epoch_restored": 14,
    "test_used": False
}

with open(
    os.path.join(
        SAVE_DIR_V3C,
        "v3c_metrics.json"
    ),
    "w"
) as f:
    json.dump(
        metrics_v3c,
        f,
        indent=4
    )

print("✅ CNN-LSTM V3-C guardada.")

✅ CNN-LSTM V3-C guardada.


In [ ]:
# ============================================================
# COMPARACIÓN DEFINITIVA DE CANDIDATAS CNN-LSTM V3
# CRITERIO PREDEFINIDO: MAXIMUM VALIDATION MACRO-F1
# TEST SET NOT USED
# ============================================================

import os
import json
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/CAN_IDS_V3"

paths = {
    "V3-A": os.path.join(
        BASE_DIR,
        "V3A",
        "v3a_metrics.json"
    ),

    "V3-B": os.path.join(
        BASE_DIR,
        "V3B",
        "v3b_metrics.json"
    ),

    "V3-C": os.path.join(
        BASE_DIR,
        "V3C",
        "v3c_metrics.json"
    )
}


# ============================================================
# CARGAR MÉTRICAS
# ============================================================

records = []

for model_name, path in paths.items():

    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"No se encontró:\n{path}"
        )

    with open(
        path,
        "r"
    ) as f:
        metrics = json.load(f)

    records.append({

        "Model":
            model_name,

        "Parameters":
            metrics.get(
                "parameters",
                None
            ),

        "Validation sequences":
            metrics[
                "validation_sequences"
            ],

        "Accuracy (%)":
            metrics[
                "accuracy"
            ] * 100,

        "Macro Precision (%)":
            metrics[
                "macro_precision"
            ] * 100,

        "Macro Recall (%)":
            metrics[
                "macro_recall"
            ] * 100,

        "Macro F1 (%)":
            metrics[
                "macro_f1"
            ] * 100,

        "Weighted F1 (%)":
            metrics[
                "weighted_f1"
            ] * 100,

        "Errors":
            metrics[
                "misclassifications"
            ],

        "Training Time (s)":
            metrics[
                "training_time_seconds"
            ],

        "Epochs":
            metrics[
                "epochs_executed"
            ],

        "Test used":
            metrics.get(
                "test_used",
                False
            )
    })


comparison_v3 = pd.DataFrame(
    records
)


# ============================================================
# ORDENAR ÚNICAMENTE POR MACRO-F1 DE VALIDATION
# ============================================================

comparison_v3 = (
    comparison_v3
    .sort_values(
        by="Macro F1 (%)",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n")
print("=" * 120)
print("CNN-LSTM V3 — FINAL VALIDATION MODEL SELECTION")
print("=" * 120)

print(
    comparison_v3.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# GANADORA SEGÚN CRITERIO PREDEFINIDO
# ============================================================

winner = comparison_v3.iloc[0]

print("\n")
print("=" * 70)
print("SELECTED CNN-LSTM ARCHITECTURE")
print("=" * 70)

print(
    "Selected model:",
    winner["Model"]
)

print(
    f"Validation Macro-F1: "
    f"{winner['Macro F1 (%)']:.6f}%"
)

print(
    f"Validation Accuracy: "
    f"{winner['Accuracy (%)']:.6f}%"
)

print(
    "Misclassifications:",
    int(winner["Errors"])
)

print(
    "\n✅ Architecture selected exclusively "
    "using Validation Macro-F1."
)

print(
    "✅ Test set was not used for V3 model selection."
)



CNN-LSTM V3 — FINAL VALIDATION MODEL SELECTION
Model   Parameters  Validation sequences  Accuracy (%)  Macro Precision (%)  Macro Recall (%)  Macro F1 (%)  Weighted F1 (%)  Errors  Training Time (s)  Epochs  Test used
 V3-C 49060.000000                 36907     99.986452            99.987945         99.982884     99.985413        99.986452       5         999.555440      18      False
 V3-A          NaN                 36907     99.981033            99.984988         99.971025     99.978004        99.981032       7         884.933220      12      False
 V3-B 49956.000000                 36907     99.975614            99.975072         99.966248     99.970657        99.975615       9         608.589795      12      False


SELECTED CNN-LSTM ARCHITECTURE
Selected model: V3-C
Validation Macro-F1: 99.985413%
Validation Accuracy: 99.986452%
Misclassifications: 5

✅ Architecture selected exclusively using Validation Macro-F1.
✅ Test set was not used for V3 model selection.


In [ ]:
# ============================================================
# FREEZE FINAL V3 SELECTION
# ============================================================

import json
import os
from datetime import datetime

selected_model = comparison_v3.iloc[0]

selection_manifest = {

    "selected_architecture":
        selected_model["Model"],

    "selection_criterion":
        "Maximum validation macro-F1",

    "validation_macro_f1_percent":
        float(
            selected_model[
                "Macro F1 (%)"
            ]
        ),

    "validation_accuracy_percent":
        float(
            selected_model[
                "Accuracy (%)"
            ]
        ),

    "validation_errors":
        int(
            selected_model[
                "Errors"
            ]
        ),

    "candidate_models": [
        "V3-A",
        "V3-B",
        "V3-C"
    ],

    "test_used_for_selection":
        False,

    "architecture_frozen":
        True
}


manifest_path = os.path.join(
    BASE_DIR,
    "FINAL_V3_SELECTION.json"
)


with open(
    manifest_path,
    "w"
) as f:

    json.dump(
        selection_manifest,
        f,
        indent=4
    )


print(
    "✅ Final V3 selection frozen:"
)

print(
    manifest_path
)

✅ Final V3 selection frozen:
/content/drive/MyDrive/CAN_IDS_V3/FINAL_V3_SELECTION.json


In [ ]:
# ============================================================
# VERIFY FINAL V3 FREEZE
# ============================================================

import os
import json

manifest_path = (
    "/content/drive/MyDrive/"
    "CAN_IDS_V3/"
    "FINAL_V3_SELECTION.json"
)

assert os.path.isfile(manifest_path)

with open(manifest_path, "r") as f:
    final_selection = json.load(f)

print("==============================================")
print("FINAL CNN-LSTM V3 ARCHITECTURE")
print("==============================================")

print(
    "Selected architecture:",
    final_selection["selected_architecture"]
)

print(
    "Selection criterion:",
    final_selection["selection_criterion"]
)

print(
    "Validation Macro-F1:",
    final_selection["validation_macro_f1_percent"]
)

print(
    "Validation Accuracy:",
    final_selection["validation_accuracy_percent"]
)

print(
    "Validation Errors:",
    final_selection["validation_errors"]
)

print(
    "Test used for selection:",
    final_selection["test_used_for_selection"]
)

print(
    "Architecture frozen:",
    final_selection["architecture_frozen"]
)

print("\n✅ CNN-LSTM V3-C formally frozen.")

FINAL CNN-LSTM V3 ARCHITECTURE
Selected architecture: V3-C
Selection criterion: Maximum validation macro-F1
Validation Macro-F1: 99.98541328306275
Validation Accuracy: 99.98645243449752
Validation Errors: 5
Test used for selection: False
Architecture frozen: True

✅ CNN-LSTM V3-C formally frozen.


In [ ]:
print("V3-A parameters:", model_v3a.count_params())

V3-A parameters: 78884
